<a href="https://colab.research.google.com/github/Rohan-1103/Data-Science/blob/main/DL/keras_hyperparameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [139]:
import pandas as pd
import numpy as np

### Reading the data

In [140]:
df = pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


### Checking effect of each feature on Outcome column

In [141]:
df.corr()["Outcome"]

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


### X and Y
### Applying train test split

In [142]:
x = df.iloc[:, :-1]
y = df.iloc[:, -1]

In [143]:
x

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63
764,2,122,70,27,0,36.8,0.340,27
765,5,121,72,23,112,26.2,0.245,30
766,1,126,60,0,0,30.1,0.349,47


### Data Preprocessing

In [144]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [145]:
x = scaler.fit_transform(x)

In [146]:
x.shape

(768, 8)

### Train test split

In [147]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=43)

### Neural Network Model

In [148]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Input, Dropout

In [149]:
model = Sequential()

In [150]:
model.add(Input(shape=(8,)))   # 👈 define input here
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [151]:
model.fit(x_train, y_train, batch_size = 32, epochs=50, validation_data=(x_test, y_test))

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - accuracy: 0.4121 - loss: 0.7397 - val_accuracy: 0.5519 - val_loss: 0.6935
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5586 - loss: 0.6704 - val_accuracy: 0.6429 - val_loss: 0.6429
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6840 - loss: 0.6199 - val_accuracy: 0.7078 - val_loss: 0.6045
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7410 - loss: 0.5831 - val_accuracy: 0.7338 - val_loss: 0.5795
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7524 - loss: 0.5541 - val_accuracy: 0.7532 - val_loss: 0.5620
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7606 - loss: 0.5330 - val_accuracy: 0.7597 - val_loss: 0.5489
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7590 - loss: 0.5169 - val_accuracy: 0.7662 - val_loss: 0.5401
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7590 - loss: 0.5035 - val_accuracy: 0.7727 - val_loss

## Problem:
 - Have to decide layers.
 - Have to decide nodes.
 - Have to decide activation.
 - Have to decide loss.
 - Have to decide optimizer.

 ## Solution:
  - Automate using Keras-Tuner performing hyperparameter tuning

In [152]:
!pip install keras-tuner
import kerastuner as kt

### Steps:
  1. Make a function.
  2. Make a tuner object.
  3. Pass tuner object to function.

### 1. Make a function - build_model(hp)
 - To tune best optimizer

In [153]:
# Selecting best optimizer
def build_model(hp):
  model = Sequential()
  model.add(Input(shape=(8, )))
  model.add(Dense(32, activation='relu'))
  model.add(Dense(1, activation='sigmoid'))

  optimizer = hp.Choice('optimizer', values = ['adam', 'sgd', 'rmsprop', 'adadelta'])
  model.compile(optimizer=optimizer, loss = 'binary_crossentropy', metrics=['accuracy'])

  return model

### Make a tuner object and pass it to Function via RandomSearch

In [154]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [155]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

In [156]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [157]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [158]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [159]:
model.fit(x_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 121ms/step - accuracy: 0.7541 - loss: 0.5444 - val_accuracy: 0.7468 - val_loss: 0.5388
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7736 - loss: 0.5182 - val_accuracy: 0.7468 - val_loss: 0.5211
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7818 - loss: 0.4999 - val_accuracy: 0.7597 - val_loss: 0.5113
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7834 - loss: 0.4877 - val_accuracy: 0.7662 - val_loss: 0.5063
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7818 - loss: 0.4787 - val_accuracy: 0.7662 - val_loss: 0.5016
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7834 - loss: 0.4734 - val_accuracy: 0.7662 - val_loss: 0.4994
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7850 - loss: 0.4676 - val_accuracy: 0.7662 - val_loss: 0.4980
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7883 - loss: 0.4641 - val_accura

- To tune best number of neurons

In [160]:
def build_model(hp):
  model = Sequential()

  units = hp.Int('units', min_value = 8, max_value = 128, step=8)

  model.add(Input(shape=(8, )))
  model.add(Dense(units = units, activation='relu'))
  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer='rmsprop',  loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [161]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5, directory='mydir', project_name = 'Hyperparameter Tune')

Reloading Tuner from mydir/Hyperparameter Tune/tuner0.json


In [162]:
tuner.search(x_train, y_train, epochs = 5, validation_data = (x_test, y_test))

In [163]:
tuner.get_best_hyperparameters()[0].values

{'units': 80}

In [164]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [165]:
model.fit(x_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - accuracy: 0.7720 - loss: 0.4703 - val_accuracy: 0.7922 - val_loss: 0.5143
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.4606 - val_accuracy: 0.7922 - val_loss: 0.5123
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.4560 - val_accuracy: 0.7922 - val_loss: 0.5149
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7818 - loss: 0.4522 - val_accuracy: 0.7922 - val_loss: 0.5135
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7866 - loss: 0.4497 - val_accuracy: 0.7857 - val_loss: 0.5128
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7834 - loss: 0.4464 - val_accuracy: 0.7922 - val_loss: 0.5149
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7899 - loss: 0.4452 - val_accuracy: 0.7792 - val_loss: 0.5174
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7866 - loss: 0.4428 - val_accuracy: 0.76

### To tune the number of hidden layers

In [166]:
def build_model(hp):
  model = Sequential()

  model.add(Input(shape=(8, )))
  model.add(Dense(72, activation = 'relu'))

  for i in range(hp.Int('num_layers', min_value = 1, max_value = 10)):
    model.add(Dense(72, activation='relu'))

  model.add(Dense(1, activation = 'sigmoid'))
  model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [167]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=3, directory = 'mydir', project_name='num_layers')

Reloading Tuner from mydir/num_layers/tuner0.json


In [168]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

In [169]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 2}

In [170]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [171]:
model.fit(x_train, y_train, epochs = 100, initial_epoch=6, validation_data=(x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.7769 - loss: 0.4672 - val_accuracy: 0.7727 - val_loss: 0.4873
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7915 - loss: 0.4470 - val_accuracy: 0.7662 - val_loss: 0.5005
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8013 - loss: 0.4390 - val_accuracy: 0.7597 - val_loss: 0.5059
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7980 - loss: 0.4375 - val_accuracy: 0.7532 - val_loss: 0.5145
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8029 - loss: 0.4274 - val_accuracy: 0.7338 - val_loss: 0.5103
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8094 - loss: 0.4233 - val_accuracy: 0.7597 - val_loss: 0.5344
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8143 - loss: 0.4178 - val_accuracy: 0.7338 - val_loss: 0.5193
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8160 - loss: 0.4078 - val_accuracy: 0.73

### tuning multiple params

In [172]:
import keras
from keras import layers, Sequential

def build_model(hp):
    model = Sequential()

    # 1. Define number of layers to tune
    num_layers = hp.Int('num_layers', min_value=1, max_value=10)

    for i in range(num_layers):
        # 2. Add Dense layer with tunable units and activation
        # Note: 'input_shape' is only needed for the very first layer
        if i == 0:
            model.add(layers.Dense(
                units=hp.Int(f'units_{i}', min_value=8, max_value=128, step=8),
                activation=hp.Choice(f'activation_{i}', values=['relu', 'tanh', 'sigmoid']),
                input_shape=(8,)
            ))
        else:
            model.add(layers.Dense(
                units=hp.Int(f'units_{i}', min_value=8, max_value=128, step=8),
                activation=hp.Choice(f'activation_{i}', values=['relu', 'tanh', 'sigmoid'])
            ))

        # 3. Add Tunable Dropout after each Dense layer
        model.add(layers.Dropout(
            rate=hp.Choice(f'dropout_{i}', values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
        ))

    # 4. Final Output Layer for binary classification
    model.add(layers.Dense(1, activation='sigmoid'))

    # 5. Tunable Optimizer
    model.compile(
        optimizer=hp.Choice('optimizer', values=['rmsprop', 'adam', 'sgd', 'nadam', 'adadelta']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [173]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=3, directory='mydir', project_name='final1')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [174]:
tuner.search(x_train, y_train, epochs = 5, validation_data = (x_test, y_test))

Trial 3 Complete [00h 00m 13s]
val_accuracy: 0.649350643157959

Best val_accuracy So Far: 0.649350643157959
Total elapsed time: 00h 00m 34s


In [175]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 8,
 'units_0': 104,
 'activation_0': 'sigmoid',
 'dropout_0': 0.5,
 'optimizer': 'adam',
 'units_1': 8,
 'activation_1': 'relu',
 'dropout_1': 0.1,
 'units_2': 8,
 'activation_2': 'relu',
 'dropout_2': 0.1,
 'units_3': 8,
 'activation_3': 'relu',
 'dropout_3': 0.1,
 'units_4': 8,
 'activation_4': 'relu',
 'dropout_4': 0.1,
 'units_5': 8,
 'activation_5': 'relu',
 'dropout_5': 0.1,
 'units_6': 8,
 'activation_6': 'relu',
 'dropout_6': 0.1,
 'units_7': 8,
 'activation_7': 'relu',
 'dropout_7': 0.1}

In [176]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 38 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [177]:
model.fit(x_train, y_train, epochs = 200, initial_epoch=6, validation_data=(x_test, y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 12s 228ms/step - accuracy: 0.6433 - loss: 0.6809 - val_accuracy: 0.6494 - val_loss: 0.6765
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6531 - loss: 0.6712 - val_accuracy: 0.6494 - val_loss: 0.6620
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6466 - loss: 0.6656 - val_accuracy: 0.6494 - val_loss: 0.6491
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6498 - loss: 0.6469 - val_accuracy: 0.6494 - val_loss: 0.6393
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6498 - loss: 0.6589 - val_accuracy: 0.6494 - val_loss: 0.6344
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6498 - loss: 0.6454 - val_accuracy: 0.6494 - val_loss: 0.6255
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6482 - loss: 0.6343 - val_accuracy: 0.6494 - val_loss: 0.6126
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6498 - loss: 0.6397 - val_accuracy: